In [0]:
from pyspark.sql import functions as F

# Union inpatient and outpatient
bronze_inpatient_table = "health.default.bronze_claims_inpatient"
bronze_outpatient_table = "health.default.bronze_claims_outpatient"
silver_claims_table = "health.default.silver_claims_unified"

df_all_claims = spark.table(bronze_inpatient_table) \
    .select(
        "BeneID", "ClaimID", "ClaimStartDt", "ClaimEndDt", "Provider",
        "InscClaimAmtReimbursed", "AttendingPhysician", "claim_type",
        *[f"ClmDiagnosisCode_{i}" for i in range(1, 11)]
    ) \
    .unionByName(
        spark.table(bronze_outpatient_table).select(
            "BeneID", "ClaimID", "ClaimStartDt", "ClaimEndDt", "Provider",
            "InscClaimAmtReimbursed", "AttendingPhysician", "claim_type",
            *[f"ClmDiagnosisCode_{i}" for i in range(1, 11)]
        ),
        allowMissingColumns=True
    )

# Add derived features
df_all_claims = df_all_claims \
    .withColumn("claim_duration_days", 
                F.datediff(F.col("ClaimEndDt"), F.col("ClaimStartDt"))) \
    .withColumn("claim_year", F.year("ClaimStartDt")) \
    .withColumn("claim_month", F.month("ClaimStartDt")) \
    .withColumn("claim_day_of_week", F.dayofweek("ClaimStartDt"))

# Write to silver table
df_all_claims.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("claim_year", "claim_type") \
    .saveAsTable(silver_claims_table)

print(f"Created {silver_claims_table}")

In [0]:
from pyspark.sql.window import Window

silver_provider_stats_table = "health.default.silver_provider_statistics"

# Calculate provider statistics
df_provider_stats = df_all_claims \
    .groupBy("Provider") \
    .agg(
        F.count("ClaimID").alias("total_claims"),
        F.countDistinct("BeneID").alias("unique_patients"),
        F.sum("InscClaimAmtReimbursed").alias("total_reimbursement"),
        F.avg("InscClaimAmtReimbursed").alias("avg_claim_amount"),
        F.stddev("InscClaimAmtReimbursed").alias("claim_amount_stddev"),
        F.max("InscClaimAmtReimbursed").alias("max_claim_amount"),
        F.avg("claim_duration_days").alias("avg_claim_duration"),
        F.countDistinct("AttendingPhysician").alias("unique_physicians"),
        F.sum(F.when(F.col("claim_type") == "inpatient", 1).otherwise(0)).alias("inpatient_claims"),
        F.sum(F.when(F.col("claim_type") == "outpatient", 1).otherwise(0)).alias("outpatient_claims")
    ) \
    .withColumn("claims_per_patient", F.col("total_claims") / F.col("unique_patients")) \
    .withColumn("inpatient_ratio", F.col("inpatient_claims") / F.col("total_claims"))

# Statistical outlier detection (Z-score based)
stats = df_provider_stats.select(
    F.mean("total_reimbursement").alias("mean_reimbursement"),
    F.stddev("total_reimbursement").alias("std_reimbursement"),
    F.mean("claims_per_patient").alias("mean_cpp"),
    F.stddev("claims_per_patient").alias("std_cpp")
).first()

df_provider_stats = df_provider_stats \
    .withColumn("reimbursement_zscore", 
                (F.col("total_reimbursement") - F.lit(stats['mean_reimbursement'])) / 
                F.lit(stats['std_reimbursement'])) \
    .withColumn("cpp_zscore",
                (F.col("claims_per_patient") - F.lit(stats['mean_cpp'])) / 
                F.lit(stats['std_cpp'])) \
    .withColumn("is_outlier",
                F.when((F.abs(F.col("reimbursement_zscore")) > 3) | 
                       (F.abs(F.col("cpp_zscore")) > 3), 1).otherwise(0))

# Write to silver table
df_provider_stats.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_provider_stats_table)

print(f"Created {silver_provider_stats_table}")

In [0]:
silver_beneficiary_profiles_table = "health.default.silver_beneficiary_profiles"
bronze_beneficiaries_table = "health.default.bronze_beneficiaries"

# Aggregate by beneficiary
df_bene_claims = df_all_claims \
    .groupBy("BeneID") \
    .agg(
        F.count("ClaimID").alias("total_claims"),
        F.countDistinct("Provider").alias("unique_providers"),
        F.sum("InscClaimAmtReimbursed").alias("total_costs"),
        F.avg("InscClaimAmtReimbursed").alias("avg_claim_cost"),
        F.countDistinct("AttendingPhysician").alias("unique_physicians"),
        F.sum(F.when(F.col("claim_type") == "inpatient", 1).otherwise(0)).alias("inpatient_visits"),
        F.sum(F.when(F.col("claim_type") == "outpatient", 1).otherwise(0)).alias("outpatient_visits")
    )

# Join with beneficiary demographics
df_bene_enriched = spark.table(bronze_beneficiaries_table) \
    .join(df_bene_claims, "BeneID", "left") \
    .fillna(0, subset=["total_claims", "unique_providers", "total_costs"])

# Calculate age
df_bene_enriched = df_bene_enriched \
    .withColumn("age", 
                F.round(F.months_between(F.current_date(), F.col("DOB")) / 12)) \
    .withColumn("is_deceased", F.when(F.col("DOD").isNotNull(), 1).otherwise(0))

# Chronic condition count
chronic_cols = [
    "ChronicCond_Alzheimer", "ChronicCond_Heartfailure", "ChronicCond_KidneyDisease",
    "ChronicCond_Cancer", "ChronicCond_ObstrPulmonary", "ChronicCond_Depression",
    "ChronicCond_Diabetes", "ChronicCond_IschemicHeart", "ChronicCond_Osteoporasis",
    "ChronicCond_rheumatoidarthritis", "ChronicCond_stroke"
]

df_bene_enriched = df_bene_enriched \
    .withColumn("chronic_condition_count", 
                sum([F.col(c) for c in chronic_cols]))

# Risk stratification
df_bene_enriched = df_bene_enriched \
    .withColumn("risk_score",
                (F.col("chronic_condition_count") * 10) + 
                (F.col("age") * 0.5) + 
                (F.col("inpatient_visits") * 5)) \
    .withColumn("risk_category",
                F.when(F.col("risk_score") > 100, "High")
                 .when(F.col("risk_score") > 50, "Medium")
                 .otherwise("Low"))

df_bene_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_beneficiary_profiles_table)

print(f"Created {silver_beneficiary_profiles_table}")

In [0]:
# Flatten diagnosis codes
from pyspark.sql.functions import explode, array
from pyspark.sql.window import Window

silver_provider_diagnosis_patterns_table = "health.default.silver_provider_diagnosis_patterns"

df_diagnoses = df_all_claims.select(
    "Provider",
    "BeneID",
    "ClaimID",
    "InscClaimAmtReimbursed",
    explode(array(*[f"ClmDiagnosisCode_{i}" for i in range(1, 11)])).alias("diagnosis_code")
).filter(F.col("diagnosis_code").isNotNull())

# Top diagnoses by provider
df_provider_diagnoses = df_diagnoses \
    .groupBy("Provider", "diagnosis_code") \
    .agg(
        F.count("*").alias("diagnosis_count"),
        F.avg("InscClaimAmtReimbursed").alias("avg_cost")
    ) \
    .withColumn("rank", F.dense_rank().over(
        Window.partitionBy("Provider").orderBy(F.desc("diagnosis_count"))
    )) \
    .filter(F.col("rank") <= 10)

# Calculate diagnosis diversity
df_provider_diversity = df_diagnoses \
    .groupBy("Provider") \
    .agg(
        F.countDistinct("diagnosis_code").alias("unique_diagnoses"),
        F.count("*").alias("total_diagnoses")
    ) \
    .withColumn("diagnosis_diversity_ratio", 
                F.col("unique_diagnoses") / F.col("total_diagnoses"))

df_provider_diversity.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_provider_diagnosis_patterns_table)

print(f"Created {silver_provider_diagnosis_patterns_table}")